pour 30000 patient ca passe sur gpu, pour 120000 passer sur CPU, donc penser a changer

In [ ]:
# ==============================================================================
# CLUSTERING PIPELINE — HDBSCAN + UMAP + GOWER - MINMAX SCALER
# ==============================================================================

# ==============================================================================
# 0. CONFIGURATION
# ==============================================================================

# ── Paths ──────────────────────────────────────────────────────────────────────
CSV_PATH   = "df_final_binaire_imputed.csv"
OUTPUT_DIR = "Results/Regular_clustering/Full_dataset/With_counts/minmax_scaler"

# ── GPU ────────────────────────────────────────────────────────────────────────
GPU_DEVICE_ID = 1

# ── UMAP (réduction dimensionnelle avant HDBSCAN) ─────────────────────────────
UMAP_N_NEIGHBORS  = 30
UMAP_MIN_DIST     = 0.0
UMAP_N_COMPONENTS = 10
UMAP_RANDOM_STATE = 42

# ── UMAP / t-SNE (visualisation uniquement) ───────────────────────────────────
VIZ_N_NEIGHBORS = 30
VIZ_MIN_DIST    = 0.1
VIZ_TSNE_PERP   = 30
VIZ_TSNE_ITER   = 1500

# ── HDBSCAN fixed run ─────────────────────────────────────────────────────────
HDBSCAN_MIN_CLUSTER_SIZE = 1000
HDBSCAN_MIN_SAMPLES      = HDBSCAN_MIN_CLUSTER_SIZE   # = mcs (recommandation sklearn)
HDBSCAN_CLUSTER_METHOD   = "eom"

# ── mcs Sweep ─────────────────────────────────────────────────────────────────
SWEEP_VALUES      = list(range(200, 2500, 100))
SWEEP_MIN_SAMPLES = {mcs: mcs for mcs in SWEEP_VALUES}  # min_samples = mcs

# ── Sweep min_samples (étape 2, après avoir fixé mcs optimal) ─────────────────
BEST_MCS        = 1000
SWEEP_MS_VALUES = [1] + list(range(10, 100, 20)) + list(range(100, BEST_MCS + 1, 50))

# ── Combined score weights ─────────────────────────────────────────────────────
W_SILHOUETTE = 0.4
W_STABILITY  = 0.3
W_OUTLIER    = 0.3

# ── Poids scenario_2 ──────────────────────────────────────────────────────────
WEIGHT_HOSP_S2 = 3.0

# ── Poids scenario_3 (Gower) ──────────────────────────────────────────────────
WEIGHT_BIO_VEINOUS      = 1 / 29
WEIGHT_IMAGING_DETAILED = 1 / 12
WEIGHT_HOSP_S3          = 5.0

# ── Categorisation bins ────────────────────────────────────────────────────────
BIO_BINS    = [-1, 0, 1, 2, 10]
BIO_LABELS  = ["0", "1", "2", "3+"]
IMAG_BINS   = [-1, 0, 1, 2, 10]
IMAG_LABELS = ["0", "1", "2", "3+"]


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

import os
import logging
import time
import gower
import hdbscan
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
import torch
import umap
import umap.umap_ as umap_reduce
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances, silhouette_score

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

# ── GPU setup ──────────────────────────────────────────────────────────────────
torch.cuda.set_device(GPU_DEVICE_ID)
torch.cuda.set_per_process_memory_fraction(0.5, device=GPU_DEVICE_ID)
print(f"GPU : {torch.cuda.get_device_name(GPU_DEVICE_ID)}")
print(f"Mémoire disponible : {torch.cuda.get_device_properties(GPU_DEVICE_ID).total_memory / 1e9:.0f} GB")


# ==============================================================================
# 2. COLUMN DEFINITIONS
# ==============================================================================

IMAGING_COLS_BOOL = {
    "has_ultrasound", "has_ct_scan", "has_xray", "has_mri",
}

IMAGING_COLS_DETAILED = {
    "ultrasound_1", "ultrasound_2",
    "ct_scan_1", "ct_scan_2", "ct_scan_3",
    "xray_1", "xray_2", "xray_3",
    "mri_1", "mri_2",
}

BIO_VEINOUS = {
    "is_hemoglobine", "is_leucocytes", "is_formule_leuco",
    "is_urea", "is_creatinine", "is_sodium", "is_potassium",
    "is_platelets", "is_pt", "is_aptt", "is_calcium", "is_ck",
    "is_lactates", "is_troponine", "is_bnp", "is_ckmb", "is_ddimer",
    "is_crp", "is_pct", "is_alat", "is_asat", "is_bili_total",
    "is_lipase", "is_alp", "is_iron", "is_ferritin",
    "is_calcium_ionized", "is_aXa_aIIa", "is_fibrinogen",
}

BIO_EXAMS = {
    "has_blood_test", "has_culture",
    "has_lumbar_puncture", "has_blood_gas",
}

PROCEDURE_COLS   = {"had_ekg"}
DISPOSITION_COLS = {
    "hospitalization", "observation_unit", "inter_facility_transfer",
}

COLS_QUANTI    = ["imaging_exam_count", "bio_exam_count"]
COLS_BINARY    = list(IMAGING_COLS_BOOL | BIO_EXAMS | PROCEDURE_COLS | BIO_VEINOUS | DISPOSITION_COLS)
COLS_MULTI_CAT = list(IMAGING_COLS_DETAILED)

SCENARIOS = {
    "scenario_2": list(
        IMAGING_COLS_BOOL | BIO_EXAMS | PROCEDURE_COLS | DISPOSITION_COLS
    ) + COLS_QUANTI,

    "scenario_3": list(
        IMAGING_COLS_BOOL | BIO_EXAMS | PROCEDURE_COLS | DISPOSITION_COLS
        | IMAGING_COLS_DETAILED | BIO_VEINOUS
    ) + COLS_QUANTI,
}


# ==============================================================================
# 3. DATA LOADING & PREPROCESSING
# ==============================================================================

def load_and_preprocess(csv_path: str = CSV_PATH) -> pd.DataFrame:
    df = pd.read_csv(csv_path, low_memory=False)

    df["bio_exam_cat"] = pd.cut(
        df["bio_exam_count"], bins=BIO_BINS, labels=BIO_LABELS
    )
    df["imaging_exam_cat"] = pd.cut(
        df["imaging_exam_count"], bins=IMAG_BINS, labels=IMAG_LABELS
    )

    for col in ("bio_exam_count", "imaging_exam_count"):
        dist = (
            df[col].value_counts(dropna=False).sort_index()
            .rename_axis(col).reset_index(name="n")
        )
        dist["pct"] = (dist["n"] / dist["n"].sum() * 100).round(1)
        log.info(f"\n{col} distribution:\n{dist.to_string(index=False)}")

    return df


# ==============================================================================
# 4. DISTANCE MATRIX — GPU (Hamming + Manhattan)
# ==============================================================================

def compute_distance_matrix_gpu(
    df_sub:        pd.DataFrame,
    binary_cols:   list,
    cat_cols:      list,
    quanti_cols:   list,
    weight_hosp:   float = 1.0,
    weight_quanti: float = 1.0,
    device_id:     int   = GPU_DEVICE_ID,
) -> np.ndarray:
    """
    Pairwise distance matrix on GPU (PyTorch).
    Binary/cat → Hamming | Quanti → Manhattan MinMax-normalised.
    """
    device = torch.device(f"cuda:{device_id}")
    n      = len(df_sub)
    D      = torch.zeros((n, n), device=device, dtype=torch.float32)

    for col in binary_cols:
        vals = torch.tensor(
            df_sub[col].values, dtype=torch.float32, device=device
        ).unsqueeze(1)
        d = torch.cdist(vals, vals, p=1)
        w = weight_hosp if col == "hospitalization" else 1.0
        D += d * w

    for col in cat_cols:
        codes = torch.tensor(
            df_sub[col].cat.codes.values, dtype=torch.float32, device=device
        ).unsqueeze(1)
        D += (codes != codes.T).float()

    if quanti_cols:
        q        = df_sub[quanti_cols].values.astype("float32")
        q_scaled = MinMaxScaler().fit_transform(q).astype("float32")  # ← MinMax
        q_tensor = torch.tensor(q_scaled, dtype=torch.float32, device=device)
        D       += torch.cdist(q_tensor, q_tensor, p=1) * weight_quanti

    D.fill_diagonal_(0)
    return D.cpu().numpy().astype(np.float64)


# ==============================================================================
# 5. CLUSTERING — run_hdbscan
# ==============================================================================

def run_hdbscan(
    df:               pd.DataFrame,
    scenario_name:    str,
    run_label:        str,
    distance_metric:  str,
    weight_hosp:      float = 1.0,
    weight_quanti:    float = 1.0,
    gower_weights:    dict  = None,
    min_cluster_size: int   = HDBSCAN_MIN_CLUSTER_SIZE,
    min_samples:      int   = HDBSCAN_MIN_SAMPLES,      # = mcs par défaut
):
    """
    Run HDBSCAN clustering.
    Retourne : df_sub, D, fit_input, labels, clusterer
               fit_input = D pour scenario_2, embedding UMAP pour scenario_3
    """
    sc_dir = os.path.join(OUTPUT_DIR, run_label)
    os.makedirs(sc_dir, exist_ok=True)

    # ── Sélection des colonnes ─────────────────────────────────────────────────
    cols   = [c for c in SCENARIOS[scenario_name] if c in df.columns]
    df_sub = df[cols].copy()

    if scenario_name == "scenario_3":
        essential = [c for c in cols if c not in IMAGING_COLS_DETAILED and c not in BIO_VEINOUS]
        df_sub    = df_sub.dropna(subset=essential)
    else:
        df_sub = df_sub.dropna()

    idx = df_sub.index

    # ── Détection des types de colonnes ───────────────────────────────────────
    quanti_cols = [c for c in COLS_QUANTI if c in df_sub.columns]
    binary_cols = [
        c for c in df_sub.columns
        if c not in quanti_cols and set(df_sub[c].dropna().unique()) <= {0, 1}
    ]
    cat_cols = [
        c for c in df_sub.columns
        if c not in binary_cols and c not in quanti_cols
    ]
    for col in cat_cols:
        df_sub[col] = df_sub[col].astype("category")

    log.info(
        f"[{run_label}] binary={len(binary_cols)} | "
        f"cat={len(cat_cols)} | quanti={len(quanti_cols)} | n={len(df_sub)}"
    )

    # ── Matrice de distances ───────────────────────────────────────────────────
    t0       = time.time()
    col_list = list(df_sub.columns)
    n_vars   = len(col_list)

    if distance_metric == "gower":
        df_sub_gower = df_sub.copy()
        for col in df_sub_gower.select_dtypes(include="integer").columns:
            df_sub_gower[col] = df_sub_gower[col].astype("float64")
        for col in df_sub_gower.select_dtypes(include="category").columns:
            df_sub_gower[col] = df_sub_gower[col].astype("object")

        weights = np.ones(n_vars)
        if gower_weights:
            for col, w in gower_weights.items():
                if col in col_list:
                    weights[col_list.index(col)] = w
        weights = weights / weights.sum() * n_vars

        D = gower.gower_matrix(df_sub_gower, weight=weights)
        D = D.astype(np.float64)
        np.fill_diagonal(D, 0)
        fit_input      = D
        hdbscan_metric = "precomputed"

    elif distance_metric == "precomputed":
        D = compute_distance_matrix_gpu(
            df_sub,
            binary_cols   = binary_cols,
            cat_cols      = cat_cols,
            quanti_cols   = quanti_cols,
            weight_hosp   = weight_hosp,
            weight_quanti = weight_quanti,
        )
        fit_input      = D
        hdbscan_metric = "precomputed"

    else:
        raise ValueError(f"distance_metric doit être 'gower' ou 'precomputed', pas '{distance_metric}'.")

    log.info(f"[{run_label}] Distance matrix : {time.time()-t0:.1f}s")

    # ── UMAP (scenario_3 uniquement) ──────────────────────────────────────────
    if scenario_name == "scenario_3":
        t0 = time.time()
        reducer = umap_reduce.UMAP(
            n_neighbors  = UMAP_N_NEIGHBORS,
            min_dist     = UMAP_MIN_DIST,
            n_components = UMAP_N_COMPONENTS,
            metric       = "precomputed",
            random_state = UMAP_RANDOM_STATE,
        )
        emb = reducer.fit_transform(D)
        np.save(os.path.join(sc_dir, "umap_embedding.npy"), emb)
        log.info(f"[{run_label}] UMAP embedding shape={emb.shape} — {time.time()-t0:.1f}s")
        fit_input      = emb
        hdbscan_metric = "euclidean"

    # ── HDBSCAN ───────────────────────────────────────────────────────────────
    t0 = time.time()
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size         = min_cluster_size,
        min_samples              = min_samples,
        metric                   = hdbscan_metric,
        cluster_selection_method = HDBSCAN_CLUSTER_METHOD,
        gen_min_span_tree        = True,
    ).fit(fit_input)

    log.info(f"[{run_label}] HDBSCAN : {time.time()-t0:.1f}s")

    labels     = clusterer.labels_
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = (labels == -1).sum()

    log.info(
        f"[{run_label}] clusters={n_clusters} | "
        f"noise={n_noise} ({100*n_noise/len(labels):.1f}%)"
    )

    # ── Export CSV ─────────────────────────────────────────────────────────────
    df_out = df.loc[idx].copy()
    df_out["cluster"] = labels
    df_out.to_csv(os.path.join(sc_dir, f"clustering_mcs{min_cluster_size}.csv"), index=False)

    return df_sub, D, fit_input, labels, clusterer  # ← fit_input ajouté


# ==============================================================================
# 6. SWEEP mcs
# ==============================================================================

def run_hdbscan_sweep(
    fit_input:    np.ndarray,
    metric:       str,
    sweep_values: list = SWEEP_VALUES,
) -> pd.DataFrame:
    """
    Sweep over min_cluster_size with min_samples = mcs (recommandation sklearn).
    """
    np.fill_diagonal(fit_input, 0) if metric == "precomputed" else None
    results = []

    for mcs in sweep_values:
        ms = SWEEP_MIN_SAMPLES[mcs]   # dynamique : min_samples = mcs

        clusterer = hdbscan.HDBSCAN(
            min_cluster_size         = mcs,
            min_samples              = ms,
            metric                   = metric,
            cluster_selection_method = HDBSCAN_CLUSTER_METHOD,
            gen_min_span_tree        = True,
        ).fit(fit_input)

        labels          = clusterer.labels_
        unique_clusters = np.unique(labels[labels >= 0])
        n_clusters      = len(unique_clusters)

        sil = (
            silhouette_score(fit_input, labels, metric=metric)
            if n_clusters >= 2 else np.nan
        )
        stability = (
            float(np.mean(clusterer.cluster_persistence_))
            if len(clusterer.cluster_persistence_) > 0 else np.nan
        )

        results.append({
            "min_cluster_size": mcs,
            "min_samples":      ms,
            "n_clusters":       n_clusters,
            "silhouette":       sil,
            "outlier_rate":     float(np.mean(labels == -1)),
            "stability":        stability,
            "labels":           labels,
            "probabilities":    clusterer.probabilities_,
            "cluster_sizes":    {c: int(np.sum(labels == c)) for c in unique_clusters},
            "clusterer":        clusterer,
        })

    return pd.DataFrame(results)


def compute_combined_score(
    df_sweep: pd.DataFrame,
    w_sil:    float = W_SILHOUETTE,
    w_stab:   float = W_STABILITY,
    w_out:    float = W_OUTLIER,
) -> pd.DataFrame:
    df = df_sweep.copy()
    df["silhouette"] = df["silhouette"].fillna(0)
    df["stability"]  = df["stability"].fillna(0)

    def _minmax(s):
        mn, mx = s.min(), s.max()
        return (s - mn) / (mx - mn) if mx != mn else s * 0

    df["combined_score"] = (
        w_sil * _minmax(df["silhouette"])
        + w_stab * _minmax(df["stability"])
        - w_out * df["outlier_rate"]
    )
    return df


# ==============================================================================
# 7. SWEEP min_samples
# ==============================================================================

def run_min_samples_sweep(
    fit_input:    np.ndarray,
    metric:       str,
    best_mcs:     int  = BEST_MCS,
    sweep_ms:     list = SWEEP_MS_VALUES,
    out_dir:      str  = OUTPUT_DIR,
    run_label:    str  = "min_samples_sweep",
) -> pd.DataFrame:
    """
    Sweep de min_samples avec mcs fixé.
    À lancer APRÈS run_hdbscan_sweep() et choix du mcs optimal.
    """
    np.fill_diagonal(fit_input, 0) if metric == "precomputed" else None
    results = []

    for ms in sweep_ms:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size         = best_mcs,
            min_samples              = ms,
            metric                   = metric,
            cluster_selection_method = HDBSCAN_CLUSTER_METHOD,
            gen_min_span_tree        = True,
        ).fit(fit_input)

        labels          = clusterer.labels_
        unique_clusters = np.unique(labels[labels >= 0])
        n_clusters      = len(unique_clusters)

        sil = (
            silhouette_score(fit_input, labels, metric=metric)
            if n_clusters >= 2 else np.nan
        )
        stability = (
            float(np.mean(clusterer.cluster_persistence_))
            if len(clusterer.cluster_persistence_) > 0 else np.nan
        )

        results.append({
            "min_samples":      ms,
            "min_cluster_size": best_mcs,
            "n_clusters":       n_clusters,
            "silhouette":       sil,
            "outlier_rate":     float(np.mean(labels == -1)),
            "stability":        stability,
            "labels":           labels,
            "clusterer":        clusterer,
        })

        log.info(
            f"[{run_label}] ms={ms:5d} | clusters={n_clusters} | "
            f"bruit={np.mean(labels==-1)*100:.1f}% | "
            f"sil={sil:.3f if not np.isnan(sil) else 'nan'}"
        )

    df_results = pd.DataFrame(results)

    out = os.path.join(out_dir, run_label)
    os.makedirs(out, exist_ok=True)

    fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharex=True)
    axes[0].plot(df_results["min_samples"], df_results["n_clusters"], marker="o", color="tab:blue")
    axes[0].set_ylabel("Nombre de clusters")
    axes[0].set_title(f"Sweep min_samples — {run_label}\n(mcs fixé à {best_mcs})")
    axes[0].grid(True)
    axes[1].plot(df_results["min_samples"], df_results["outlier_rate"] * 100, marker="s", color="tab:red")
    axes[1].set_ylabel("Outlier rate (%)")
    axes[1].grid(True)
    axes[2].plot(df_results["min_samples"], df_results["silhouette"], marker="d", color="tab:green")
    axes[2].set_ylabel("Silhouette score")
    axes[2].set_xlabel("min_samples")
    axes[2].grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(out, "min_samples_sweep.png"), dpi=150)
    plt.close()

    df_results.drop(columns=["labels", "clusterer"]) \
              .to_csv(os.path.join(out, f"ms_sweep_mcs{best_mcs}.csv"), index=False)

    log.info(f"[{run_label}] Sweep min_samples terminé → {out}")
    return df_results


# ==============================================================================
# 8. HELPER — ms sweep pour un mcs donné
# ==============================================================================

def run_ms_sweep_for_mcs(
    df:               pd.DataFrame,
    scenario_name:    str,
    run_label:        str,
    distance_metric:  str,
    best_mcs:         int,
    gower_weights:    dict  = None,
    weight_hosp:      float = 1.0,
    weight_quanti:    float = 1.0,
) -> tuple:
    """
    Pour un mcs donné :
    1. Recalcule D + fit_input
    2. Lance le sweep de min_samples dessus
    """
    df_sub, D, fit_input, labels, clusterer = run_hdbscan(
        df,
        scenario_name    = scenario_name,
        run_label        = run_label,
        distance_metric  = distance_metric,
        weight_hosp      = weight_hosp,
        weight_quanti    = weight_quanti,
        gower_weights    = gower_weights,
        min_cluster_size = best_mcs,
    )

    metric   = "euclidean" if scenario_name == "scenario_3" else "precomputed"
    sweep_ms = list(range(50, best_mcs + 1, 50))

    log.info(
        f"[{run_label}] mcs={best_mcs} → "
        f"clusters={len(set(labels))-(1 if -1 in labels else 0)} | "
        f"outliers={np.mean(labels==-1)*100:.1f}%"
    )

    df_ms = run_min_samples_sweep(
        fit_input = fit_input,
        metric    = metric,
        best_mcs  = best_mcs,
        sweep_ms  = sweep_ms,
        out_dir   = OUTPUT_DIR,
        run_label = run_label,
    )

    return df_sub, D, fit_input, labels, clusterer, df_ms


# ==============================================================================
# 9. VISUALISATION — SWEEP CURVES
# ==============================================================================

def plot_sweep_curves(df_sweep, title, out_dir, filename_prefix):
    os.makedirs(out_dir, exist_ok=True)
    x      = df_sweep["min_cluster_size"]
    y_sil  = df_sweep["silhouette"]
    y_stab = df_sweep["stability"]
    y_comb = df_sweep["combined_score"]

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.plot(x, y_sil, color="tab:blue", marker="o", label="Silhouette")
    ax1.set_xlabel("min_cluster_size")
    ax1.set_ylabel("Silhouette score", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")
    ax2 = ax1.twinx()
    ax2.plot(x, y_stab, color="tab:red", marker="s", label="Stability")
    ax2.set_ylabel("Mean stability", color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")
    plt.title(title)
    fig.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{filename_prefix}_silhouette_stability.png"), dpi=150)
    plt.close()

    plt.figure(figsize=(10, 6))
    plt.plot(x, y_comb, color="tab:green", marker="d", linewidth=2)
    plt.xlabel("min_cluster_size")
    plt.ylabel("Combined score")
    plt.title("Combined score (silhouette + stability - outliers)")
    plt.grid(True)
    plt.savefig(os.path.join(out_dir, f"{filename_prefix}_combined_score.png"), dpi=150)
    plt.close()


# ==============================================================================
# 10. VISUALISATION — HDBSCAN TREES
# ==============================================================================

def plot_hdbscan_trees(
    clusterer: hdbscan.HDBSCAN,
    run_label: str,
    out_dir:   str,
) -> None:
    os.makedirs(out_dir, exist_ok=True)

    # ── 1. Condensed Tree ─────────────────────────────────────────────────────
    try:
        fig, ax = plt.subplots(figsize=(14, 7))
        clusterer.condensed_tree_.plot(
            select_clusters = True,
            axis            = ax,
            colorbar        = True,
        )
        ax.set_title(f"Condensed Tree — {run_label}\n(axe Y = λ = 1/distance | plus haut = plus stable)", fontsize=11)
        ax.set_xlabel("Points / clusters")
        ax.set_ylabel("λ (stabilité)")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, "condensed_tree.png"), dpi=150)
        plt.close()
        log.info(f"[{run_label}] condensed_tree.png sauvegardé")
    except Exception as e:
        log.warning(f"[{run_label}] Condensed tree : {e}")

    # ── 2. Single Linkage Tree ────────────────────────────────────────────────
    try:
        fig, ax = plt.subplots(figsize=(14, 7))
        clusterer.single_linkage_tree_.plot(axis=ax)
        ax.set_title(f"Single Linkage Tree — {run_label}\n(hiérarchie brute complète avant élagage)", fontsize=11)
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, "single_linkage_tree.png"), dpi=150)
        plt.close()
        log.info(f"[{run_label}] single_linkage_tree.png sauvegardé")
    except Exception as e:
        log.warning(f"[{run_label}] Single linkage tree : {e}")

    # ── 3. MST ────────────────────────────────────────────────────────────────
    try:
        mst = clusterer.minimum_spanning_tree_
        if mst is None:
            log.info(f"[{run_label}] MST non disponible (metric='precomputed')")
        else:
            fig, ax = plt.subplots(figsize=(14, 7))
            mst.plot(edge_cmap="viridis", edge_alpha=0.6, node_size=10, axis=ax)
            ax.set_title(f"Minimum Spanning Tree — {run_label}", fontsize=11)
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "minimum_spanning_tree.png"), dpi=150)
            plt.close()
            log.info(f"[{run_label}] minimum_spanning_tree.png sauvegardé")
    except Exception as e:
        log.warning(f"[{run_label}] MST : {e}")

    # ── 4. Cluster Persistence ────────────────────────────────────────────────
    try:
        persistence = clusterer.cluster_persistence_
        if len(persistence) > 0:
            fig, ax = plt.subplots(figsize=(max(6, len(persistence)), 4))
            colors  = ["#e05c2e" if p == max(persistence) else "#aec6cf" for p in persistence]
            ax.bar(range(len(persistence)), persistence, color=colors)
            ax.set_xlabel("Cluster ID")
            ax.set_ylabel("Persistence (stabilité)")
            ax.set_title(f"Cluster Persistence — {run_label}\n(rouge = cluster le plus stable)", fontsize=11)
            ax.set_xticks(range(len(persistence)))
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, "cluster_persistence.png"), dpi=150)
            plt.close()
            log.info(f"[{run_label}] cluster_persistence.png sauvegardé")
    except Exception as e:
        log.warning(f"[{run_label}] Cluster persistence : {e}")


# ==============================================================================
# 11. VISUALISATION — CLUSTER PROFILES
# ==============================================================================

def plot_cluster_heatmap(df_sub, labels, title, out_dir, filename):
    os.makedirs(out_dir, exist_ok=True)
    df_prof            = df_sub.copy()
    df_prof["cluster"] = labels
    df_prof            = df_prof[df_prof["cluster"] != -1]

    if df_prof.empty:
        log.warning(f"[{title}] Aucun cluster trouvé — heatmap ignorée.")
        return pd.DataFrame()

    numeric_cols = [c for c in df_prof.select_dtypes(include=["number"]).columns.tolist() if c != "cluster"]
    profile      = df_prof[numeric_cols + ["cluster"]].groupby("cluster").mean().round(3)

    if profile.empty or profile.shape[0] == 0 or profile.shape[1] == 0:
        log.warning(f"[{title}] Profile vide — heatmap ignorée.")
        return profile

    n_cols = profile.shape[1]
    n_rows = profile.shape[0]
    plt.figure(figsize=(max(8, n_cols * 0.8), max(4, n_rows * 0.6)))
    sns.heatmap(profile, annot=True, cmap="YlOrRd", fmt=".2f",
                annot_kws={"size": max(6, min(10, 80 // n_cols))})
    plt.title(title)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    path = os.path.join(out_dir, filename)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"Saved: {path}")
    return profile


def plot_cluster_dendrogram(profile, title, out_dir, filename):
    os.makedirs(out_dir, exist_ok=True)
    Z               = linkage(profile.values, method="ward")
    variables       = profile.columns.tolist()
    cluster_vectors = {i: profile.iloc[i].values for i in range(len(profile))}

    plt.figure(figsize=(10, 5))
    dendrogram(Z, labels=profile.index.astype(str))
    plt.gca().grid(False)
    plt.title(title)

    for i, (c1, c2, dist, _) in enumerate(Z):
        c1, c2  = int(c1), int(c2)
        v1, v2  = cluster_vectors[c1], cluster_vectors[c2]
        top_var = variables[np.argmax(np.abs(v1 - v2))]
        cluster_vectors[len(cluster_vectors)] = (v1 + v2) / 2
        plt.text(i + 1, dist, top_var, rotation=45, fontsize=8, va="bottom", ha="center")

    plt.tight_layout()
    path = os.path.join(out_dir, filename)
    plt.savefig(path, dpi=150)
    plt.close()
    log.info(f"Saved: {path}")


# ==============================================================================
# 12. VISUALISATION — UMAP & t-SNE
# ==============================================================================

def build_cluster_palette(labels: np.ndarray) -> dict:
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
    n_clusters      = len(unique_clusters)
    palette         = sns.color_palette("tab20", n_clusters) if n_clusters <= 20 \
                      else sns.color_palette("hsv", n_clusters)
    color_map       = {c: palette[i] for i, c in enumerate(unique_clusters)}
    color_map[-1]   = "lightgrey"
    return color_map


def plot_umap_2d(X, labels, title, out_dir, filename, metric="precomputed"):
    os.makedirs(out_dir, exist_ok=True)
    emb = umap.UMAP(n_components=2, n_neighbors=VIZ_N_NEIGHBORS,
                    min_dist=VIZ_MIN_DIST, metric=metric).fit_transform(X)
    color_map       = build_cluster_palette(labels)
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
    plt.figure(figsize=(9, 7))
    mask_noise = labels == -1
    if mask_noise.any():
        plt.scatter(emb[mask_noise, 0], emb[mask_noise, 1], c="lightgrey",
                    s=8, linewidth=0, label="Outliers (-1)", zorder=1, alpha=0.5)
    for c in unique_clusters:
        mask = labels == c
        plt.scatter(emb[mask, 0], emb[mask, 1], color=color_map[c],
                    s=10, linewidth=0, label=f"C{c}", zorder=2, alpha=0.8)
    plt.title(title)
    plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc="upper left", markerscale=2, fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, filename), dpi=150)
    plt.close()
    log.info(f"Saved: {os.path.join(out_dir, filename)}")
    return color_map


def plot_umap_3d_html(X, labels, title, out_dir, filename_html,
                      metric="precomputed", color_map: dict = None):
    os.makedirs(out_dir, exist_ok=True)
    emb = umap.UMAP(n_components=3, n_neighbors=VIZ_N_NEIGHBORS,
                    min_dist=VIZ_MIN_DIST, metric=metric).fit_transform(X)
    if color_map is None:
        color_map = build_cluster_palette(labels)
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
    def to_hex(c):
        return "#d3d3d3" if c == "lightgrey" else mcolors.to_hex(c)
    df_plot = pd.DataFrame({"x": emb[:, 0], "y": emb[:, 1], "z": emb[:, 2],
                             "label": [str(l) for l in labels]})
    df_plot["order"] = df_plot["label"].apply(lambda x: -1 if x == "-1" else int(x))
    df_plot = df_plot.sort_values("order")
    fig = px.scatter_3d(df_plot, x="x", y="y", z="z", color="label",
                        color_discrete_map={str(c): to_hex(color_map[c]) for c in list(unique_clusters) + [-1]},
                        title=title, opacity=0.8,
                        category_orders={"label": ["-1"] + [str(c) for c in unique_clusters]})
    fig.update_traces(marker=dict(size=3))
    fig.update_layout(legend_title_text="Cluster")
    fig.write_html(os.path.join(out_dir, filename_html), include_plotlyjs="cdn")
    log.info(f"Saved: {filename_html}")


def plot_tsne_2d(X, labels, title, out_dir, filename,
                 metric="precomputed", color_map: dict = None):
    os.makedirs(out_dir, exist_ok=True)
    init = umap.UMAP(n_components=2, n_neighbors=VIZ_N_NEIGHBORS,
                     min_dist=VIZ_MIN_DIST, metric=metric).fit_transform(X)
    emb  = TSNE(n_components=2, perplexity=VIZ_TSNE_PERP, learning_rate="auto",
                init=init, metric=metric, max_iter=VIZ_TSNE_ITER).fit_transform(X)
    if color_map is None:
        color_map = build_cluster_palette(labels)
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
    plt.figure(figsize=(9, 7))
    mask_noise = labels == -1
    if mask_noise.any():
        plt.scatter(emb[mask_noise, 0], emb[mask_noise, 1], c="lightgrey",
                    s=8, linewidth=0, label="Outliers (-1)", zorder=1, alpha=0.5)
    for c in unique_clusters:
        mask = labels == c
        plt.scatter(emb[mask, 0], emb[mask, 1], color=color_map[c],
                    s=10, linewidth=0, label=f"C{c}", zorder=2, alpha=0.8)
    plt.title(title)
    plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc="upper left", markerscale=2, fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, filename), dpi=150)
    plt.close()
    log.info(f"Saved: {os.path.join(out_dir, filename)}")


def plot_tsne_3d_html(X, labels, title, out_dir, filename_html,
                      metric="precomputed", color_map: dict = None):
    os.makedirs(out_dir, exist_ok=True)
    init = umap.UMAP(n_components=3, n_neighbors=VIZ_N_NEIGHBORS,
                     min_dist=VIZ_MIN_DIST, metric=metric).fit_transform(X)
    emb  = TSNE(n_components=3, perplexity=VIZ_TSNE_PERP, learning_rate="auto",
                init=init, metric=metric, max_iter=VIZ_TSNE_ITER).fit_transform(X)
    if color_map is None:
        color_map = build_cluster_palette(labels)
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
    def to_hex(c):
        return "#d3d3d3" if c == "lightgrey" else mcolors.to_hex(c)
    df_plot = pd.DataFrame({"x": emb[:, 0], "y": emb[:, 1], "z": emb[:, 2],
                             "label": [str(l) for l in labels]})
    fig = px.scatter_3d(df_plot, x="x", y="y", z="z", color="label",
                        color_discrete_map={str(c): to_hex(color_map[c]) for c in list(unique_clusters) + [-1]},
                        title=title, opacity=0.8,
                        category_orders={"label": ["-1"] + [str(c) for c in unique_clusters]})
    fig.update_traces(marker=dict(size=3))
    fig.update_layout(legend_title_text="Cluster")
    fig.write_html(os.path.join(out_dir, filename_html), include_plotlyjs="cdn")
    log.info(f"Saved: {filename_html}")


# ==============================================================================
# 13. VISUALISATION — OUTLIERS
# ==============================================================================

def describe_outliers_internal(df_sub, labels, run_label, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    df_work            = df_sub.copy()
    df_work["cluster"] = labels
    df_noise           = df_work[df_work["cluster"] == -1]
    df_clustered       = df_work[df_work["cluster"] != -1]
    n_noise            = len(df_noise)
    n_total            = len(df_work)
    log.info(f"[{run_label}] Outliers : {n_noise} / {n_total} ({100*n_noise/n_total:.1f}%)")
    if n_noise == 0:
        log.info("Aucun outlier.")
        return
    numeric_cols = [c for c in df_sub.select_dtypes(include="number").columns]
    compare = pd.DataFrame({
        "outliers":  df_noise[numeric_cols].mean().round(3),
        "clustered": df_clustered[numeric_cols].mean().round(3),
    })
    compare["diff"] = (compare["outliers"] - compare["clustered"]).round(3)
    compare = compare.sort_values("diff", key=abs, ascending=False)
    fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 0.8), 4))
    sns.heatmap(compare[["outliers", "clustered"]].T, annot=True, fmt=".2f",
                cmap="YlOrRd", ax=ax, annot_kws={"size": 8})
    ax.set_title(f"Outliers vs clustered — {run_label}")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "outliers_internal_heatmap.png"), dpi=150, bbox_inches="tight")
    plt.close()
    fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 0.8), 5))
    colors  = ["tab:red" if v > 0 else "tab:blue" for v in compare["diff"]]
    ax.bar(compare.index, compare["diff"], color=colors, alpha=0.8)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel("Différence (outliers - clustered)")
    ax.set_title(f"Différences outliers vs clustered — {run_label}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "outliers_internal_diff.png"), dpi=150)
    plt.close()
    compare.to_csv(os.path.join(out_dir, "outliers_internal_compare.csv"))
    log.info(f"[{run_label}] Description outliers terminée.")


# ==============================================================================
# 14. POSTPROCESSING
# ==============================================================================

def run_postprocessing(
    df_sub:    pd.DataFrame,
    D:         np.ndarray,
    fit_input: np.ndarray,
    labels:    np.ndarray,
    clusterer: hdbscan.HDBSCAN,
    run_label: str,
    out_dir:   str,
    metric:    str = "precomputed",
):
    t_start = time.time()
    os.makedirs(out_dir, exist_ok=True)

    # ── 1. Trees HDBSCAN ──────────────────────────────────────────────────────
    plot_hdbscan_trees(
        clusterer = clusterer,
        run_label = run_label,
        out_dir   = os.path.join(out_dir, "hdbscan_trees"),
    )

    # ── 2. Sweep mcs ──────────────────────────────────────────────────────────
    df_sweep = run_hdbscan_sweep(fit_input, metric=metric)
    df_sweep = compute_combined_score(df_sweep)
    df_sweep.drop(columns=["labels", "probabilities", "cluster_sizes", "clusterer"]) \
            .to_csv(os.path.join(out_dir, "sweep.csv"), index=False)
    plot_sweep_curves(df_sweep, title=f"Sweep — {run_label}",
                      out_dir=out_dir, filename_prefix=run_label)

    # ── 3. Heatmap + dendrogramme ─────────────────────────────────────────────
    profile = plot_cluster_heatmap(df_sub, labels,
                                   title=f"Cluster profiles — {run_label}",
                                   out_dir=out_dir, filename="heatmap.png")
    if len(profile) >= 2:
        plot_cluster_dendrogram(profile, title=f"Dendrogram — {run_label}",
                                out_dir=out_dir, filename="dendrogram.png")

    # ── 4. UMAP + t-SNE ───────────────────────────────────────────────────────
    viz_input  = fit_input if metric == "euclidean" else D
    viz_metric = metric

    color_map = plot_umap_2d(viz_input, labels, title=f"UMAP 2D — {run_label}",
                             out_dir=out_dir, filename="umap2d.png", metric=viz_metric)
    plot_umap_3d_html(viz_input, labels, title=f"UMAP 3D — {run_label}",
                      out_dir=out_dir, filename_html="umap3d.html",
                      metric=viz_metric, color_map=color_map)
    plot_tsne_2d(viz_input, labels, title=f"t-SNE 2D — {run_label}",
                 out_dir=out_dir, filename="tsne2d.png",
                 metric=viz_metric, color_map=color_map)
    plot_tsne_3d_html(viz_input, labels, title=f"t-SNE 3D — {run_label}",
                      out_dir=out_dir, filename_html="tsne3d.html",
                      metric=viz_metric, color_map=color_map)

    # ── 5. Outliers ───────────────────────────────────────────────────────────
    describe_outliers_internal(df_sub=df_sub, labels=labels,
                               run_label=run_label,
                               out_dir=os.path.join(out_dir, "outliers_internal"))

    log.info(f"[{run_label}] Postprocessing total : {time.time()-t_start:.1f}s")


# ==============================================================================
# 15. PIPELINE COMPLET
# ==============================================================================

def run_full_pipeline(df: pd.DataFrame):
    runs = [
        dict(scenario_name="scenario_2", run_label="s2_noweights",
             distance_metric="precomputed", weight_hosp=1.0, weight_quanti=1.0, gower_weights=None),
        dict(scenario_name="scenario_2", run_label="s2_hosp3x",
             distance_metric="precomputed", weight_hosp=WEIGHT_HOSP_S2, weight_quanti=1.0, gower_weights=None),
        dict(scenario_name="scenario_3", run_label="s3_noweights",
             distance_metric="gower", weight_hosp=1.0, weight_quanti=1.0, gower_weights=None),
        dict(scenario_name="scenario_3", run_label="s3_catdown",
             distance_metric="gower", weight_hosp=1.0, weight_quanti=1.0,
             gower_weights={**{col: WEIGHT_BIO_VEINOUS for col in BIO_VEINOUS},
                            **{col: WEIGHT_IMAGING_DETAILED for col in IMAGING_COLS_DETAILED}}),
        dict(scenario_name="scenario_3", run_label="s3_catdown_hosp5x",
             distance_metric="gower", weight_hosp=1.0, weight_quanti=1.0,
             gower_weights={**{col: WEIGHT_BIO_VEINOUS for col in BIO_VEINOUS},
                            **{col: WEIGHT_IMAGING_DETAILED for col in IMAGING_COLS_DETAILED},
                            "hospitalization": WEIGHT_HOSP_S3}),
    ]

    for run in runs:
        t_run     = time.time()
        run_label = run["run_label"]
        out_dir   = os.path.join(OUTPUT_DIR, run_label)
        scenario  = run["scenario_name"]

        log.info(f"\n{'='*60}\nRun : {run_label}\n{'='*60}")

        df_sub, D, fit_input, labels, clusterer = run_hdbscan(
            df,
            scenario_name   = scenario,
            run_label       = run_label,
            distance_metric = run["distance_metric"],
            weight_hosp     = run["weight_hosp"],
            weight_quanti   = run["weight_quanti"],
            gower_weights   = run["gower_weights"],
        )

        sweep_metric = "euclidean" if scenario == "scenario_3" else "precomputed"

        run_postprocessing(
            df_sub    = df_sub,
            D         = D,
            fit_input = fit_input,
            labels    = labels,
            clusterer = clusterer,
            run_label = run_label,
            out_dir   = out_dir,
            metric    = sweep_metric,
        )

        log.info(f"[{run_label}] ✅ Run complet : {time.time()-t_run:.1f}s")


# ==============================================================================
# 16. ENTRY POINT
# ==============================================================================

if __name__ == "__main__":
    df = load_and_preprocess(CSV_PATH)
    run_full_pipeline(df)

In [1]:
# # ==============================================================================
# # CLUSTERING PIPELINE — HDBSCAN + UMAP + GOWER
# # ==============================================================================
# #
# # Install dependencies:
# #   pip install pandas numpy scikit-learn hdbscan umap-learn gower
# #              seaborn matplotlib scipy plotly torch
# #
# # ==============================================================================
#
#
# # ==============================================================================
# # 0. CONFIGURATION  — edit only this block
# # ==============================================================================
#
# # ── Paths ──────────────────────────────────────────────────────────────────────
# CSV_PATH   = "df_final_binaire_imputed.csv"
# OUTPUT_DIR = "Results/Regular_clustering/Full_dataset/With_counts/minmax_scaler"
#
#
# # ── GPU ────────────────────────────────────────────────────────────────────────
# GPU_DEVICE_ID = 1   # GPU 1 — libre sur le serveur
#
# # ── UMAP (réduction dimensionnelle avant HDBSCAN) ─────────────────────────────
# UMAP_N_NEIGHBORS  = 30
# UMAP_MIN_DIST     = 0.0
# UMAP_N_COMPONENTS = 10
# UMAP_RANDOM_STATE = 42
#
# # ── UMAP / t-SNE (visualisation uniquement) ───────────────────────────────────
# VIZ_N_NEIGHBORS = 30
# VIZ_MIN_DIST    = 0.1 # je peux monter jusqu'à 0.5 pour une visualisation plus "globale" (moins de petits groupes serrés)
# VIZ_TSNE_PERP   = 30
# VIZ_TSNE_ITER   = 1500
#
# # ── HDBSCAN ────────────────────────────────────────────────────────────────────
# HDBSCAN_MIN_CLUSTER_SIZE = 1000
# HDBSCAN_MIN_SAMPLES      = 10
# HDBSCAN_CLUSTER_METHOD   = "eom"   # "eom" | "leaf"
#
# # ── Sweep ──────────────────────────────────────────────────────────────────────
# SWEEP_VALUES = list(range(500, 2200, 100))
#
# # ── Combined score weights ─────────────────────────────────────────────────────
# W_SILHOUETTE = 0.4
# W_STABILITY  = 0.3
# W_OUTLIER    = 0.3
#
# # ── Poids scenario_2 (precomputed Hamming+Manhattan) ──────────────────────────
# # Utilisés uniquement dans les runs avec pondération de hospitalization
# WEIGHT_HOSP_S2 = 3.0   # hospitalization × 3 dans scenario_2
#
# # ── Poids scenario_3 (Gower) — correction technique ──────────────────────────
# # Chaque groupe détaillé pèse ~1 au lieu de dominer par le nombre de variables
# WEIGHT_BIO_VEINOUS      = 1 / 29   # 29 vars × (1/29) = 1.0 au total
# WEIGHT_IMAGING_DETAILED = 1 / 12   # 12 vars × (1/12) = 1.0 au total
# # Surpondération hospitalization pour le run confirmatoire
# WEIGHT_HOSP_S3 = 5.0
#
# # ── Categorisation bins ────────────────────────────────────────────────────────
# BIO_BINS    = [-1, 0, 1, 2, 10]
# BIO_LABELS  = ["0", "1", "2", "3+"]
# IMAG_BINS   = [-1, 0, 1, 2, 10]
# IMAG_LABELS = ["0", "1", "2", "3+"]
#
#
# # ==============================================================================
# # 1. IMPORTS
# # ==============================================================================
#
# import os
# import logging
# import time
# import gower
# import hdbscan
# import matplotlib.pyplot as plt
# import numpy as np
# import pandas as pd
# import plotly.express as px
# import seaborn as sns
# import torch
# import umap
# import umap.umap_ as umap_reduce
# from sklearn.preprocessing import StandardScaler
# from scipy.cluster.hierarchy import dendrogram, linkage
# from sklearn.manifold import TSNE
# from sklearn.metrics import pairwise_distances, silhouette_score
# from sklearn.preprocessing import MinMaxScaler # a enlever si je vois que standard scaler suffit pour les quanti
#
# log = logging.getLogger(__name__)
# logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
#
# # ── GPU setup ──────────────────────────────────────────────────────────────────
# torch.cuda.set_device(GPU_DEVICE_ID)
# torch.cuda.set_per_process_memory_fraction(0.5, device=GPU_DEVICE_ID)
# print(f"GPU : {torch.cuda.get_device_name(GPU_DEVICE_ID)}")
# print(f"Mémoire disponible : {torch.cuda.get_device_properties(GPU_DEVICE_ID).total_memory / 1e9:.0f} GB")
#
#
# # ==============================================================================
# # 2. COLUMN DEFINITIONS
# # ==============================================================================
#
# # ── Imaging ────────────────────────────────────────────────────────────────────
# IMAGING_COLS_BOOL = {
#     "has_ultrasound", "has_ct_scan", "has_xray",
#     "has_mri",
#     #"has_radio_interventional", "has_nuclear_medicine", pareil trop rare
# }
#
# IMAGING_COLS_DETAILED = {
#     "ultrasound_1", "ultrasound_2",
#     "ct_scan_1", "ct_scan_2", "ct_scan_3",
#     "xray_1", "xray_2", "xray_3",
#     "mri_1", "mri_2",
#     #"radio_interventional_1",
#     #"nuclear_medicine_1", trop rare fausse tout
# }
#
# # ── Biology ────────────────────────────────────────────────────────────────────
# BIO_VEINOUS = {
#     "is_hemoglobine", "is_leucocytes", "is_formule_leuco",
#     "is_urea", "is_creatinine", "is_sodium", "is_potassium",
#     "is_platelets", "is_pt", "is_aptt", "is_calcium", "is_ck",
#     "is_lactates", "is_troponine", "is_bnp", "is_ckmb", "is_ddimer",
#     "is_crp", "is_pct", "is_alat", "is_asat", "is_bili_total",
#     "is_lipase", "is_alp", "is_iron", "is_ferritin",
#     "is_calcium_ionized", "is_aXa_aIIa", "is_fibrinogen",
# }
#
# BIO_EXAMS = {
#     "has_blood_test", "has_culture",
#     "has_lumbar_puncture", "has_blood_gas",
# }
#
# # ── Procedures & disposition ───────────────────────────────────────────────────
# PROCEDURE_COLS   = {"had_ekg"}
# DISPOSITION_COLS = {
#     "hospitalization", "observation_unit", "inter_facility_transfer",
# }
#
# # ── Colonnes quantitatives ─────────────────────────────────────────────────────
# COLS_QUANTI = ["imaging_exam_count", "bio_exam_count"]
#
# # ── Groupes pour visualisation ─────────────────────────────────────────────────
# COLS_BINARY    = list(IMAGING_COLS_BOOL | BIO_EXAMS | PROCEDURE_COLS | BIO_VEINOUS | DISPOSITION_COLS)
# COLS_MULTI_CAT = list(IMAGING_COLS_DETAILED)
#
# # ── Scenarios ──────────────────────────────────────────────────────────────────
# SCENARIOS = {
#     # Modality-level booleans + exam counts — distance Hamming+Manhattan GPU
#     "scenario_2": list(
#         IMAGING_COLS_BOOL | BIO_EXAMS | PROCEDURE_COLS | DISPOSITION_COLS
#     ) + COLS_QUANTI,
#
#     # Full detail: modality booleans + detailed imaging/venous + counts — Gower
#     "scenario_3": list(
#         IMAGING_COLS_BOOL | BIO_EXAMS | PROCEDURE_COLS | DISPOSITION_COLS
#         | IMAGING_COLS_DETAILED | BIO_VEINOUS
#     ) + COLS_QUANTI,
# }
#
#
# # ==============================================================================
# # 3. DATA LOADING & PREPROCESSING
# # ==============================================================================
#
# def load_and_preprocess(csv_path: str = CSV_PATH) -> pd.DataFrame:
#     """Load raw CSV and add bio/imaging ordinal category columns."""
#     df = pd.read_csv(csv_path, low_memory=False)
#
#     df["bio_exam_cat"] = pd.cut(
#         df["bio_exam_count"], bins=BIO_BINS, labels=BIO_LABELS
#     )
#     df["imaging_exam_cat"] = pd.cut(
#         df["imaging_exam_count"], bins=IMAG_BINS, labels=IMAG_LABELS
#     )
#
#     for col in ("bio_exam_count", "imaging_exam_count"):
#         dist = (
#             df[col].value_counts(dropna=False).sort_index()
#             .rename_axis(col).reset_index(name="n")
#         )
#         dist["pct"] = (dist["n"] / dist["n"].sum() * 100).round(1)
#         log.info(f"\n{col} distribution:\n{dist.to_string(index=False)}")
#
#     return df
#
#
# # ==============================================================================
# # 4. DISTANCE MATRIX — GPU (Hamming + Manhattan)
# # ==============================================================================
# ### CHANGER ICI POUR CPU
# def compute_distance_matrix_gpu(
#     df_sub:        pd.DataFrame,
#     binary_cols:   list,
#     cat_cols:      list,
#     quanti_cols:   list,
#     weight_hosp:   float = 1.0,
#     weight_quanti: float = 1.0,
#     device_id:     int   = GPU_DEVICE_ID,
# ) -> np.ndarray:
#     """
#     Pairwise distance matrix on GPU (PyTorch).
#     Binary/cat → Hamming | Quanti → Manhattan MinMax-normalised.
#     Returns float64 numpy array.
#     """
#     device = torch.device(f"cuda:{device_id}")
#     n      = len(df_sub)
#     D      = torch.zeros((n, n), device=device, dtype=torch.float32)
#
#     for col in binary_cols:
#         vals = torch.tensor(
#             df_sub[col].values, dtype=torch.float32, device=device
#         ).unsqueeze(1)
#         d = torch.cdist(vals, vals, p=1)
#         w = weight_hosp if col == "hospitalization" else 1.0
#         D += d * w
#
#     for col in cat_cols:
#         codes = torch.tensor(
#             df_sub[col].cat.codes.values, dtype=torch.float32, device=device
#         ).unsqueeze(1)
#         D += (codes != codes.T).float()
#
#     if quanti_cols:
#         q = df_sub[quanti_cols].values.astype("float32")
#     # minmax scaler sklearn — sur CPU
#         q_scaled = MinMaxScaler().fit_transform(q).astype("float32")
#     # Puis on envoie sur GPU
#         q_tensor = torch.tensor(q_scaled, dtype=torch.float32, device=device)
#         D       += torch.cdist(q_tensor, q_tensor, p=1) * weight_quanti
#
#     D.fill_diagonal_(0)
#     return D.cpu().numpy().astype(np.float64)
#
# # def compute_distance_matrix(
# #     df_sub:        pd.DataFrame,
# #     binary_cols:   list,
# #     cat_cols:      list,
# #     quanti_cols:   list,
# #     weight_hosp:   float = 1.0,
# #     weight_quanti: float = 1.0,
# # ) -> np.ndarray:
# #     """
# #     Pairwise distance matrix on CPU.
# #     Binary/cat → Hamming | Quanti → Manhattan StandardScaler-normalised.
# #     Returns float64 numpy array.
# #     """
# #     n = len(df_sub)
# #     D = np.zeros((n, n), dtype=np.float64)
# #
# #     # ── Binaire → Hamming ─────────────────────────────────────────────────────
# #     for col in binary_cols:
# #         vals = df_sub[col].values.reshape(-1, 1)
# #         d    = pairwise_distances(vals, metric="hamming")
# #         w    = weight_hosp if col == "hospitalization" else 1.0
# #         D   += d * w
# #
# #     # ── Catégoriel → Hamming sur codes ────────────────────────────────────────
# #     for col in cat_cols:
# #         vals = df_sub[col].cat.codes.values.reshape(-1, 1)
# #         D   += pairwise_distances(vals, metric="hamming")
# #
# #     # ── Quantitatif → Manhattan StandardScaler ────────────────────────────────
# #     if quanti_cols:
# #         q        = df_sub[quanti_cols].values.astype("float64")
# #         q_scaled = StandardScaler().fit_transform(q)
# #         D       += pairwise_distances(q_scaled, metric="manhattan") * weight_quanti
# #
# #     np.fill_diagonal(D, 0)
# #     return D
#
#
# # ==============================================================================
# # 5. CLUSTERING — run_hdbscan
# # ==============================================================================
#
# def run_hdbscan(
#     df:               pd.DataFrame,
#     scenario_name:    str,
#     run_label:        str,          # identifiant du run ex: "s2_noweights"
#     distance_metric:  str,          # "gower" | "precomputed"
#     weight_hosp:      float = 1.0,
#     weight_quanti:    float = 1.0,
#     gower_weights:    dict  = None, # {col: weight} — surcharge par colonne pour Gower
#     min_cluster_size: int   = HDBSCAN_MIN_CLUSTER_SIZE,
#     min_samples:      int   = HDBSCAN_MIN_SAMPLES,
# ):
#     """
#     Run HDBSCAN clustering.
#
#     Parameters
#     ----------
#     gower_weights : dict optionnel {col_name: weight}
#                     Surcharge les poids Gower par colonne.
#                     Si None → poids = 1.0 partout (neutre).
#
#     Returns
#     -------
#     df_sub, D, labels, clusterer
#     """
#     sc_dir = os.path.join(OUTPUT_DIR, run_label)
#     os.makedirs(sc_dir, exist_ok=True)
#
#     # ── Sélection des colonnes ─────────────────────────────────────────────────
#     cols   = [c for c in SCENARIOS[scenario_name] if c in df.columns]
#     df_sub = df[cols].copy()
#
#     # scenario_3 : garder les NaN dans les colonnes détaillées (Gower les ignore)
#     if scenario_name == "scenario_3":
#         essential = [c for c in cols if c not in IMAGING_COLS_DETAILED and c not in BIO_VEINOUS]
#         df_sub    = df_sub.dropna(subset=essential)
#     else:
#         df_sub = df_sub.dropna()
#
#     idx = df_sub.index
#
#     # ── Détection des types de colonnes ───────────────────────────────────────
#     quanti_cols = [c for c in COLS_QUANTI if c in df_sub.columns]
#     binary_cols = [
#         c for c in df_sub.columns
#         if c not in quanti_cols and set(df_sub[c].dropna().unique()) <= {0, 1}
#     ]
#     cat_cols = [
#         c for c in df_sub.columns
#         if c not in binary_cols and c not in quanti_cols
#     ]
#     for col in cat_cols:
#         df_sub[col] = df_sub[col].astype("category")
#
#     log.info(
#         f"[{run_label}] binary={len(binary_cols)} | "
#         f"cat={len(cat_cols)} | quanti={len(quanti_cols)} | n={len(df_sub)}"
#     )
#
#      # ── Matrice de distances ───────────────────────────────────────────────────────
#     t0       = time.time()
#     col_list = list(df_sub.columns)
#     n_vars   = len(col_list)
#
#     if distance_metric == "gower":
#         df_sub = df_sub.copy()
#
#         # int → float64 (requis par Gower)
#         for col in df_sub.select_dtypes(include="integer").columns:
#             df_sub[col] = df_sub[col].astype("float64")
#
#         # category → object (Gower ne gère pas CategoricalDtype)
#         for col in df_sub.select_dtypes(include="category").columns:
#             df_sub[col] = df_sub[col].astype("object")
#
#         # Vecteur de poids — neutre par défaut
#         weights = np.ones(n_vars)
#         if gower_weights:
#             for col, w in gower_weights.items():
#                 if col in col_list:
#                     weights[col_list.index(col)] = w
#         weights = weights / weights.sum() * n_vars
#
#         D = gower.gower_matrix(df_sub, weight=weights)
#         D = D.astype(np.float64)
#         np.fill_diagonal(D, 0)
#         fit_input      = D
#         hdbscan_metric = "precomputed"
#
#     elif distance_metric == "precomputed":
#         D = compute_distance_matrix_gpu(
#             df_sub,
#             binary_cols   = binary_cols,
#             cat_cols      = cat_cols,
#             quanti_cols   = quanti_cols,
#             weight_hosp   = weight_hosp,
#             weight_quanti = weight_quanti,
#         )
#         fit_input      = D
#         hdbscan_metric = "precomputed"
#
# # A ACTIVER POUR CPU — sinon ça prend trop de temps sur GPU pour 120k patients
#         # D = compute_distance_matrix(
#         # df_sub,
#         #     binary_cols   = binary_cols,
#         #     cat_cols      = cat_cols,
#         #     quanti_cols   = quanti_cols,
#         #     weight_hosp   = weight_hosp,
#         #     weight_quanti = weight_quanti,
#         # )
#     else:
#         raise ValueError(f"distance_metric doit être 'gower' ou 'precomputed', pas '{distance_metric}'.")
#
#     log.info(f"[{run_label}] Distance matrix : {time.time()-t0:.1f}s")
#
#
#     # ── UMAP (scenario_3 uniquement) ────────────────────────────
#
#     if scenario_name in ("scenario_3"):
#         t0 = time.time()
#         reducer = umap_reduce.UMAP(
#             n_neighbors  = UMAP_N_NEIGHBORS,
#             min_dist     = UMAP_MIN_DIST,
#             n_components = UMAP_N_COMPONENTS,
#             metric       = "precomputed",
#             random_state = UMAP_RANDOM_STATE,
#         )
#         emb = reducer.fit_transform(D)
#         np.save(os.path.join(sc_dir, "umap_embedding.npy"), emb)
#         log.info(f"[{run_label}] UMAP embedding shape={emb.shape}")
#         fit_input      = emb
#         hdbscan_metric = "euclidean"
#
#         log.info(f"[{run_label}] UMAP : {time.time()-t0:.1f}s")
#
#     # ── HDBSCAN ────────────────────────────────────────────────────────────────
#     t0 = time.time()
#     clusterer = hdbscan.HDBSCAN(
#         min_cluster_size         = min_cluster_size,
#         min_samples              = min_samples,
#         metric                   = hdbscan_metric,
#         cluster_selection_method = HDBSCAN_CLUSTER_METHOD,
#         gen_min_span_tree        = True,
#     ).fit(fit_input)
#
#     log.info(f"[{run_label}] HDBSCAN : {time.time()-t0:.1f}s")
#
#     labels     = clusterer.labels_
#     n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
#     n_noise    = (labels == -1).sum()
#
#     log.info(
#         f"[{run_label}] clusters={n_clusters} | "
#         f"noise={n_noise} ({100*n_noise/len(labels):.1f}%)"
#     )
#
#     # ── Export CSV ─────────────────────────────────────────────────────────────
#     df_out = df.loc[idx].copy()
#     df_out["cluster"] = labels
#     df_out.to_csv(os.path.join(sc_dir, f"clustering_mcs{min_cluster_size}.csv"), index=False)
#
#     return df_sub, D, labels, clusterer
#
#
# # ==============================================================================
# # 6. SWEEP
# # ==============================================================================
#
# def run_hdbscan_sweep(
#     D:            np.ndarray,
#     sweep_values: list = SWEEP_VALUES,
#     metric:       str  = "precomputed",
#     min_samples:  int  = HDBSCAN_MIN_SAMPLES,
# ) -> pd.DataFrame:
#     """Sweep over min_cluster_size and collect quality metrics."""
#     np.fill_diagonal(D, 0)
#     results = []
#
#     for mcs in sweep_values:
#         clusterer = hdbscan.HDBSCAN(
#             min_cluster_size         = mcs,
#             min_samples              = min_samples,
#             metric                   = metric,
#             cluster_selection_method = HDBSCAN_CLUSTER_METHOD,
#         ).fit(D)
#
#         labels          = clusterer.labels_
#         unique_clusters = np.unique(labels[labels >= 0])
#         n_clusters      = len(unique_clusters)
#
#         sil = (
#             silhouette_score(D, labels, metric=metric)
#             if n_clusters >= 2 else np.nan
#         )
#         stability = (
#             float(np.mean(clusterer.cluster_persistence_))
#             if len(clusterer.cluster_persistence_) > 0 else np.nan
#         )
#
#         results.append({
#             "min_cluster_size": mcs,
#             "n_clusters":       n_clusters,
#             "silhouette":       sil,
#             "outlier_rate":     float(np.mean(labels == -1)),
#             "stability":        stability,
#             "labels":           labels,
#             "probabilities":    clusterer.probabilities_,
#             "cluster_sizes":    {c: int(np.sum(labels == c)) for c in unique_clusters},
#         })
#
#     return pd.DataFrame(results)
#
#
# def compute_combined_score(
#     df_sweep: pd.DataFrame,
#     w_sil:    float = W_SILHOUETTE,
#     w_stab:   float = W_STABILITY,
#     w_out:    float = W_OUTLIER,
# ) -> pd.DataFrame:
#     """Add combined_score column to sweep results."""
#     df = df_sweep.copy()
#     df["silhouette"] = df["silhouette"].fillna(0)
#     df["stability"]  = df["stability"].fillna(0)
#
#     def _minmax(s):
#         mn, mx = s.min(), s.max()
#         return (s - mn) / (mx - mn) if mx != mn else s * 0
#
#     df["combined_score"] = (
#         w_sil * _minmax(df["silhouette"])
#         + w_stab * _minmax(df["stability"])
#         - w_out * df["outlier_rate"]
#     )
#     return df
#
#
# # ==============================================================================
# # 7. VISUALISATION — SWEEP CURVES
# # ==============================================================================
#
# def plot_sweep_curves(df_sweep, title, out_dir, filename_prefix):
#     os.makedirs(out_dir, exist_ok=True)
#     x      = df_sweep["min_cluster_size"]
#     y_sil  = df_sweep["silhouette"]
#     y_stab = df_sweep["stability"]
#     y_comb = df_sweep["combined_score"]
#
#     fig, ax1 = plt.subplots(figsize=(10, 6))
#     ax1.plot(x, y_sil, color="tab:blue", marker="o", label="Silhouette")
#     ax1.set_xlabel("min_cluster_size")
#     ax1.set_ylabel("Silhouette score", color="tab:blue")
#     ax1.tick_params(axis="y", labelcolor="tab:blue")
#     ax2 = ax1.twinx()
#     ax2.plot(x, y_stab, color="tab:red", marker="s", label="Stability")
#     ax2.set_ylabel("Mean stability", color="tab:red")
#     ax2.tick_params(axis="y", labelcolor="tab:red")
#     plt.title(title)
#     fig.tight_layout()
#     path1 = os.path.join(out_dir, f"{filename_prefix}_silhouette_stability.png")
#     plt.savefig(path1, dpi=150)
#     plt.close()
#     log.info(f"Saved: {path1}")
#
#     plt.figure(figsize=(10, 6))
#     plt.plot(x, y_comb, color="tab:green", marker="d", linewidth=2)
#     plt.xlabel("min_cluster_size")
#     plt.ylabel("Combined score")
#     plt.title("Combined score (silhouette + stability - outliers)")
#     plt.grid(True)
#     path2 = os.path.join(out_dir, f"{filename_prefix}_combined_score.png")
#     plt.savefig(path2, dpi=150)
#     plt.close()
#     log.info(f"Saved: {path2}")
#
#
# # ==============================================================================
# # 8. VISUALISATION — CLUSTER PROFILES
# # ==============================================================================
#
# def plot_cluster_heatmap(df_sub, labels, title, out_dir, filename):
#     os.makedirs(out_dir, exist_ok=True)
#     df_prof            = df_sub.copy()
#     df_prof["cluster"] = labels
#     df_prof            = df_prof[df_prof["cluster"] != -1]
#
#     # ← garder uniquement les colonnes numériques pour la moyenne
#     numeric_cols = [
#         c for c in df_prof.select_dtypes(include=["number"]).columns.tolist()
#         if c != "cluster"
#     ]
#     profile      = df_prof[numeric_cols + ["cluster"]].groupby("cluster").mean().round(3)
#
#     n_cols = profile.shape[1]
#     n_rows = profile.shape[0]
#
#     plt.figure(figsize=(max(8, n_cols * 0.8), max(4, n_rows * 0.6)))
#     sns.heatmap(
#         profile, annot=True, cmap="YlOrRd", fmt=".2f",
#         annot_kws={"size": max(6, min(10, 80 // n_cols))},
#     )
#     plt.title(title)
#     plt.xticks(rotation=45, ha="right")
#     plt.tight_layout()
#     path = os.path.join(out_dir, filename)
#     plt.savefig(path, dpi=150, bbox_inches="tight")
#     plt.close()
#     log.info(f"Saved: {path}")
#     return profile
#
#
# def plot_cluster_dendrogram(profile, title, out_dir, filename):
#     os.makedirs(out_dir, exist_ok=True)
#     Z               = linkage(profile.values, method="ward")
#     variables       = profile.columns.tolist()
#     cluster_vectors = {i: profile.iloc[i].values for i in range(len(profile))}
#
#     plt.figure(figsize=(10, 5))
#     dendrogram(Z, labels=profile.index.astype(str))
#     plt.gca().grid(False)
#     plt.title(title)
#
#     for i, (c1, c2, dist, _) in enumerate(Z):
#         c1, c2  = int(c1), int(c2)
#         v1, v2  = cluster_vectors[c1], cluster_vectors[c2]
#         top_var = variables[np.argmax(np.abs(v1 - v2))]
#         cluster_vectors[len(cluster_vectors)] = (v1 + v2) / 2
#         plt.text(i + 1, dist, top_var, rotation=45, fontsize=8, va="bottom", ha="center")
#
#     plt.tight_layout()
#     path = os.path.join(out_dir, filename)
#     plt.savefig(path, dpi=150)
#     plt.close()
#     log.info(f"Saved: {path}")
#
#
# # ==============================================================================
# # 9. VISUALISATION — UMAP & t-SNE
# # ==============================================================================
# def build_cluster_palette(labels: np.ndarray) -> dict:
#     """
#     Construit un dictionnaire {cluster_id: couleur} cohérent.
#     Outliers (-1) → lightgrey
#     Clusters → couleurs distinctes tab20 ou hsv
#     """
#     unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
#     n_clusters      = len(unique_clusters)
#
#     palette = sns.color_palette("tab20", n_clusters) if n_clusters <= 20 \
#               else sns.color_palette("hsv", n_clusters)
#
#     color_map = {c: palette[i] for i, c in enumerate(unique_clusters)}
#     color_map[-1] = "lightgrey"   # outliers toujours en gris
#     return color_map
#
#
# def plot_umap_2d(X, labels, title, out_dir, filename, metric="precomputed"):
#     os.makedirs(out_dir, exist_ok=True)
#     emb       = umap.UMAP(
#         n_components=2, n_neighbors=VIZ_N_NEIGHBORS,
#         min_dist=VIZ_MIN_DIST, metric=metric,
#     ).fit_transform(X)
#
#     color_map = build_cluster_palette(labels)
#     unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
#
#     plt.figure(figsize=(9, 7))
#
#     # Outliers en premier
#     mask_noise = labels == -1
#     if mask_noise.any():
#         plt.scatter(
#             emb[mask_noise, 0], emb[mask_noise, 1],
#             c="lightgrey", s=8, linewidth=0,
#             label="Outliers (-1)", zorder=1, alpha=0.5,
#         )
#
#     # Clusters par dessus
#     for c in unique_clusters:
#         mask = labels == c
#         plt.scatter(
#             emb[mask, 0], emb[mask, 1],
#             color=color_map[c], s=10, linewidth=0,
#             label=f"C{c}", zorder=2, alpha=0.8,
#         )
#
#     plt.title(title)
#     plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1),
#                loc="upper left", markerscale=2, fontsize=8)
#     plt.tight_layout()
#     path = os.path.join(out_dir, filename)
#     plt.savefig(path, dpi=150)
#     plt.close()
#     log.info(f"Saved: {path}")
#
#     return color_map   # ← retourner pour réutilisation
#
#
# def plot_umap_3d_html(X, labels, title, out_dir, filename_html,
#                       metric="precomputed", color_map: dict = None):
#     os.makedirs(out_dir, exist_ok=True)
#     emb = umap.UMAP(
#         n_components=3, n_neighbors=VIZ_N_NEIGHBORS,
#         min_dist=VIZ_MIN_DIST, metric=metric,
#     ).fit_transform(X)
#
#     if color_map is None:
#         color_map = build_cluster_palette(labels)
#
#     unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
#
#     # Convertir couleurs matplotlib → hex pour plotly
#     def to_hex(c):
#         if c == "lightgrey":
#             return "#d3d3d3"
#         import matplotlib.colors as mcolors
#         return mcolors.to_hex(c)
#
#     # Construire le df pour plotly
#     df_plot = pd.DataFrame({
#         "x": emb[:, 0], "y": emb[:, 1], "z": emb[:, 2],
#         "label": [str(l) for l in labels],
#         "color": [to_hex(color_map[l]) for l in labels],
#     })
#
#     # Trier pour que outliers soient en dessous
#     df_plot["order"] = df_plot["label"].apply(lambda x: -1 if x == "-1" else int(x))
#     df_plot = df_plot.sort_values("order")
#
#     fig = px.scatter_3d(
#         df_plot, x="x", y="y", z="z",
#         color="label",
#         color_discrete_map={str(c): to_hex(color_map[c])
#                             for c in list(unique_clusters) + [-1]},
#         title=title, opacity=0.8,
#         category_orders={"label": ["-1"] + [str(c) for c in unique_clusters]},
#     )
#     fig.update_traces(marker=dict(size=3))
#     fig.update_layout(legend_title_text="Cluster")
#
#     path = os.path.join(out_dir, filename_html)
#     fig.write_html(path, include_plotlyjs="cdn")
#     log.info(f"Saved: {path}")
#
#
# def plot_tsne_2d(X, labels, title, out_dir, filename,
#                  metric="precomputed", color_map: dict = None):
#     os.makedirs(out_dir, exist_ok=True)
#     init = umap.UMAP(
#         n_components=2, n_neighbors=VIZ_N_NEIGHBORS,
#         min_dist=VIZ_MIN_DIST, metric=metric,
#     ).fit_transform(X)
#     emb = TSNE(
#         n_components=2, perplexity=VIZ_TSNE_PERP,
#         learning_rate="auto", init=init,
#         metric=metric, max_iter=VIZ_TSNE_ITER,
#     ).fit_transform(X)
#
#     if color_map is None:
#         color_map = build_cluster_palette(labels)
#
#     unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
#
#     plt.figure(figsize=(9, 7))
#
#     mask_noise = labels == -1
#     if mask_noise.any():
#         plt.scatter(
#             emb[mask_noise, 0], emb[mask_noise, 1],
#             c="lightgrey", s=8, linewidth=0,
#             label="Outliers (-1)", zorder=1, alpha=0.5,
#         )
#
#     for c in unique_clusters:
#         mask = labels == c
#         plt.scatter(
#             emb[mask, 0], emb[mask, 1],
#             color=color_map[c], s=10, linewidth=0,
#             label=f"C{c}", zorder=2, alpha=0.8,
#         )
#
#     plt.title(title)
#     plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1),
#                loc="upper left", markerscale=2, fontsize=8)
#     plt.tight_layout()
#     path = os.path.join(out_dir, filename)
#     plt.savefig(path, dpi=150)
#     plt.close()
#     log.info(f"Saved: {path}")
#
#
# def plot_tsne_3d_html(X, labels, title, out_dir, filename_html,
#                       metric="precomputed", color_map: dict = None):
#     os.makedirs(out_dir, exist_ok=True)
#     init = umap.UMAP(
#         n_components=3, n_neighbors=VIZ_N_NEIGHBORS,
#         min_dist=VIZ_MIN_DIST, metric=metric,
#     ).fit_transform(X)
#     emb = TSNE(
#         n_components=3, perplexity=VIZ_TSNE_PERP,
#         learning_rate="auto", init=init,
#         metric=metric, max_iter=VIZ_TSNE_ITER,
#     ).fit_transform(X)
#
#     if color_map is None:
#         color_map = build_cluster_palette(labels)
#
#     unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
#
#     def to_hex(c):
#         if c == "lightgrey":
#             return "#d3d3d3"
#         import matplotlib.colors as mcolors
#         return mcolors.to_hex(c)
#
#     df_plot = pd.DataFrame({
#         "x": emb[:, 0], "y": emb[:, 1], "z": emb[:, 2],
#         "label": [str(l) for l in labels],
#     })
#
#     fig = px.scatter_3d(
#         df_plot, x="x", y="y", z="z",
#         color="label",
#         color_discrete_map={str(c): to_hex(color_map[c])
#                             for c in list(unique_clusters) + [-1]},
#         title=title, opacity=0.8,
#         category_orders={"label": ["-1"] + [str(c) for c in unique_clusters]},
#     )
#     fig.update_traces(marker=dict(size=3))
#     fig.update_layout(legend_title_text="Cluster")
#
#     path = os.path.join(out_dir, filename_html)
#     fig.write_html(path, include_plotlyjs="cdn")
#     log.info(f"Saved: {path}")
# def describe_outliers_internal(
#     df_sub:    pd.DataFrame,
#     labels:    np.ndarray,
#     run_label: str,
#     out_dir:   str,
# ):
#     """
#     Description des outliers avec les variables internes au clustering.
#     Compare outliers vs clustered sur les variables du scenario_2.
#     """
#     os.makedirs(out_dir, exist_ok=True)
#
#     df_work            = df_sub.copy()
#     df_work["cluster"] = labels
#
#     df_noise    = df_work[df_work["cluster"] == -1]
#     df_clustered = df_work[df_work["cluster"] != -1]
#
#     n_noise   = len(df_noise)
#     n_total   = len(df_work)
#
#     log.info(
#         f"[{run_label}] Outliers : {n_noise} / {n_total} "
#         f"({100*n_noise/n_total:.1f}%)"
#     )
#
#     if n_noise == 0:
#         log.info("Aucun outlier.")
#         return
#
#     # ── Colonnes numériques uniquement ─────────────────────────────────────────
#     numeric_cols = [
#         c for c in df_sub.select_dtypes(include="number").columns
#     ]
#
#     # ── Heatmap outliers vs clustered ──────────────────────────────────────────
#     compare = pd.DataFrame({
#         "outliers":  df_noise[numeric_cols].mean().round(3),
#         "clustered": df_clustered[numeric_cols].mean().round(3),
#     })
#     compare["diff"] = (compare["outliers"] - compare["clustered"]).round(3)
#     compare = compare.sort_values("diff", key=abs, ascending=False)
#
#     fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 0.8), 4))
#     sns.heatmap(
#         compare[["outliers", "clustered"]].T,
#         annot=True, fmt=".2f", cmap="YlOrRd",
#         ax=ax, annot_kws={"size": 8}
#     )
#     ax.set_title(f"Outliers vs clustered — variables clustering — {run_label}")
#     plt.tight_layout()
#     path = os.path.join(out_dir, "outliers_internal_heatmap.png")
#     plt.savefig(path, dpi=150, bbox_inches="tight")
#     plt.close()
#     log.info(f"Saved: {path}")
#
#     # ── Barplot des différences ────────────────────────────────────────────────
#     fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 0.8), 5))
#     colors  = ["tab:red" if v > 0 else "tab:blue" for v in compare["diff"]]
#     ax.bar(compare.index, compare["diff"], color=colors, alpha=0.8)
#     ax.axhline(0, color="black", linewidth=0.8)
#     ax.set_ylabel("Différence (outliers - clustered)")
#     ax.set_title(f"Différences outliers vs clustered — {run_label}")
#     plt.xticks(rotation=45, ha="right")
#     plt.tight_layout()
#     path = os.path.join(out_dir, "outliers_internal_diff.png")
#     plt.savefig(path, dpi=150, bbox_inches="tight")
#     plt.close()
#     log.info(f"Saved: {path}")
#
#     # ── Export CSV ─────────────────────────────────────────────────────────────
#     compare.to_csv(os.path.join(out_dir, "outliers_internal_compare.csv"))
#     log.info(f"[{run_label}] Description outliers internes terminée.")
# # ==============================================================================
# # 10. HELPER — enchaîne sweep + visu pour un run donné + description des outliers
# # ==============================================================================
#
# def run_postprocessing(df_sub, D, labels, run_label, out_dir):
#     t_start = time.time()
#     """Sweep + heatmap + dendrogramme + UMAP 2D + t-SNE 2D pour un run."""
#
#     # Sweep
#     df_sweep = run_hdbscan_sweep(D, metric="precomputed")
#     df_sweep = compute_combined_score(df_sweep)
#     df_sweep.drop(columns=["labels", "probabilities", "cluster_sizes"]) \
#             .to_csv(os.path.join(out_dir, "sweep.csv"), index=False)
#     plot_sweep_curves(df_sweep, title=f"Sweep — {run_label}",
#                       out_dir=out_dir, filename_prefix=run_label)
#
#     # Heatmap + dendrogramme
#     profile = plot_cluster_heatmap(
#         df_sub, labels,
#         title=f"Cluster profiles — {run_label}",
#         out_dir=out_dir, filename="heatmap.png",
#     )
#     if len(profile) >= 2:
#         plot_cluster_dendrogram(
#             profile, title=f"Dendrogram — {run_label}",
#             out_dir=out_dir, filename="dendrogram.png",
#         )
#
#     # UMAP 2D → génère et retourne la palette
#     color_map = plot_umap_2d(
#         D, labels,
#         title=f"UMAP 2D — {run_label}",
#         out_dir=out_dir, filename="umap2d.png",
#     )
#
#     # UMAP 3D → réutilise la même palette
#     plot_umap_3d_html(
#         D, labels,
#         title=f"UMAP 3D — {run_label}",
#         out_dir=out_dir, filename_html="umap3d.html",
#         color_map=color_map,   # ← même couleurs
#     )
#
#     # t-SNE 2D → même palette
#     plot_tsne_2d(
#         D, labels,
#         title=f"t-SNE 2D — {run_label}",
#         out_dir=out_dir, filename="tsne2d.png",
#         color_map=color_map,   # ← même couleurs
#     )
#
#     # t-SNE 3D → même palette
#     plot_tsne_3d_html(
#         D, labels,
#         title=f"t-SNE 3D — {run_label}",
#         out_dir=out_dir, filename_html="tsne3d.html",
#         color_map=color_map,   # ← même couleurs
#     )
#
#     # ← ajouter ICI à la fin de run_postprocessing
#     describe_outliers_internal(
#         df_sub    = df_sub,
#         labels    = labels,
#         run_label = run_label,
#         out_dir   = os.path.join(out_dir, "outliers_internal"),
#     )
#
#
#     log.info(f"[{run_label}] postprocessing done.\n")
#
#     log.info(f"[{run_label}] Total postprocessing : {time.time()-t_start:.1f}s")
#
# # ==============================================================================
# # 11. PIPELINE COMPLET
# # ==============================================================================
#
# def run_full_pipeline(df: pd.DataFrame):
#     """
#     Runs définis :
#
#     SCENARIO 2 — precomputed (Hamming + Manhattan GPU)
#     ├── s2_noweights       : aucune pondération
#     └── s2_hosp3x          : hospitalization × 3
#
#     SCENARIO 3 — Gower
#     ├── s3_noweights       : aucune pondération
#     ├── s3_catdown         : BIO_VEINOUS + IMAGING_DETAILED sous-pondérés (correction technique)
#     └── s3_catdown_hosp5x  : idem + hospitalization × 5 (analyse confirmatoire)
#     """
#
#     runs = [
#
#         #── Scenario 2 — precomputed ───────────────────────────────────────────
#         dict(
#             scenario_name   = "scenario_2",
#             run_label       = "s2_noweights",
#             distance_metric = "precomputed",
#             weight_hosp     = 1.0,
#             weight_quanti   = 1.0,
#             gower_weights   = None,
#         ),
#         # ← s2_hosp1x supprimé, c'était identique à s2_noweights
#         dict(
#             scenario_name   = "scenario_2",
#             run_label       = "s2_hosp3x",
#             distance_metric = "precomputed",
#             weight_hosp     = WEIGHT_HOSP_S2,
#             weight_quanti   = 1.0,
#             gower_weights   = None,
#         ),
#
#
#         # ── Scenario 3 — Gower ────────────────────────────────────────────────
#         dict(
#             scenario_name   = "scenario_3",
#             run_label       = "s3_noweights",
#             distance_metric = "gower",
#             weight_hosp     = 1.0,
#             weight_quanti   = 1.0,
#             gower_weights   = None,   # ← aucune pondération
#         ),
#         dict(
#             scenario_name   = "scenario_3",
#             run_label       = "s3_catdown",
#             distance_metric = "gower",
#             weight_hosp     = 1.0,
#             weight_quanti   = 1.0,
#             gower_weights   = {
#                 # Sous-pondération technique : chaque groupe pèse ~1
#                 **{col: WEIGHT_BIO_VEINOUS      for col in BIO_VEINOUS},
#                 **{col: WEIGHT_IMAGING_DETAILED  for col in IMAGING_COLS_DETAILED},
#             },
#         ),
#         dict(
#             scenario_name   = "scenario_3",
#             run_label       = "s3_catdown_hosp5x",
#             distance_metric = "gower",
#             weight_hosp     = 1.0,   # ignoré pour gower — géré via gower_weights
#             weight_quanti   = 1.0,
#             gower_weights   = {
#                 # Sous-pondération technique
#                 **{col: WEIGHT_BIO_VEINOUS      for col in BIO_VEINOUS},
#                 **{col: WEIGHT_IMAGING_DETAILED  for col in IMAGING_COLS_DETAILED},
#                 # Surpondération confirmatoire
#                 "hospitalization": WEIGHT_HOSP_S3,
#             },
#         ),
#     ]
#
#     for run in runs:
#         t_run = time.time()
#         run_label = run["run_label"]
#         out_dir   = os.path.join(OUTPUT_DIR, run_label)
#
#         log.info(f"\n{'='*60}")
#         log.info(f"Run : {run_label}")
#         log.info(f"{'='*60}")
#
#         df_sub, D, labels, clusterer = run_hdbscan(
#             df,
#             scenario_name   = run["scenario_name"],
#             run_label       = run_label,
#             distance_metric = run["distance_metric"],
#             weight_hosp     = run["weight_hosp"],
#             weight_quanti   = run["weight_quanti"],
#             gower_weights   = run["gower_weights"],
#         )
#
#         run_postprocessing(df_sub, D, labels, run_label, out_dir)
#         log.info(f"[{run_label}] ✅ Run complet : {time.time()-t_run:.1f}s")
#
# # ==============================================================================
# # 12. ENTRY POINT
# # ==============================================================================
#
# if __name__ == "__main__":
#     df = load_and_preprocess(CSV_PATH)
#     run_full_pipeline(df)


GPU : NVIDIA A100-SXM4-80GB
Mémoire disponible : 85 GB


INFO | 
bio_exam_count distribution:
 bio_exam_count     n  pct
              0 12522 42.0
              1 12030 40.3
              2  4182 14.0
              3  1062  3.6
              4    43  0.1
INFO | 
imaging_exam_count distribution:
 imaging_exam_count     n  pct
                  0 14541 48.7
                  1 12968 43.5
                  2  2152  7.2
                  3   162  0.5
                  4    16  0.1
INFO | 
INFO | Run : s2_noweights
INFO | ============================================================
INFO | [s2_noweights] binary=12 | cat=0 | quanti=2 | n=29839
INFO | [s2_noweights] Distance matrix : 47.4s
INFO | [s2_noweights] HDBSCAN : 73.9s
INFO | [s2_noweights] clusters=12 | noise=1233 (4.1%)
INFO | Saved: Results/Regular_clustering/Full_dataset/With_counts/minmax_scaler/s2_noweights/s2_noweights_silhouette_stability.png
INFO | Saved: Results/Regular_clustering/Full_dataset/With_counts/minmax_scaler/s2_noweights/s2_noweights_combined_score.png
INFO | Saved: Res

In [2]:
def rerun_postprocessing_no_sweep(
    df:               pd.DataFrame,
    scenario_name:    str   = "scenario_2",
    run_label:        str   = "s2_noweights",
    distance_metric:  str   = "precomputed",
    min_cluster_size: int   = 1900,
    weight_hosp:      float = 1.0,
    weight_quanti:    float = 1.0,
    gower_weights:    dict  = None,
):
    """
    Relance le clustering avec un mcs spécifique et regénère
    toutes les visualisations SAUF le sweep.
    Utile pour explorer un mcs optimal sans tout relancer.
    """
    out_dir = os.path.join(OUTPUT_DIR, run_label)
    os.makedirs(out_dir, exist_ok=True)

    # ── Clustering ─────────────────────────────────────────────────────────────
    df_sub, D, labels, clusterer = run_hdbscan(
        df,
        scenario_name    = scenario_name,
        run_label        = run_label,
        distance_metric  = distance_metric,
        weight_hosp      = weight_hosp,
        weight_quanti    = weight_quanti,
        gower_weights    = gower_weights,
        min_cluster_size = min_cluster_size,
    )

    # ── Heatmap + dendrogramme ─────────────────────────────────────────────────
    profile = plot_cluster_heatmap(
        df_sub, labels,
        title    = f"Cluster profiles — {run_label} mcs={min_cluster_size}",
        out_dir  = out_dir,
        filename = f"heatmap_mcs{min_cluster_size}.png",
    )
    if len(profile) >= 2:
        plot_cluster_dendrogram(
            profile,
            title    = f"Dendrogram — {run_label} mcs={min_cluster_size}",
            out_dir  = out_dir,
            filename = f"dendrogram_mcs{min_cluster_size}.png",
        )

    # ── UMAP 2D → génère la palette ───────────────────────────────────────────
    color_map = plot_umap_2d(
        D, labels,
        title    = f"UMAP 2D — {run_label} mcs={min_cluster_size}",
        out_dir  = out_dir,
        filename = f"umap2d_mcs{min_cluster_size}.png",
    )

    # ── UMAP 3D ───────────────────────────────────────────────────────────────
    plot_umap_3d_html(
        D, labels,
        title         = f"UMAP 3D — {run_label} mcs={min_cluster_size}",
        out_dir       = out_dir,
        filename_html = f"umap3d_mcs{min_cluster_size}.html",
        color_map     = color_map,
    )

    # ── t-SNE 2D ──────────────────────────────────────────────────────────────
    plot_tsne_2d(
        D, labels,
        title     = f"t-SNE 2D — {run_label} mcs={min_cluster_size}",
        out_dir   = out_dir,
        filename  = f"tsne2d_mcs{min_cluster_size}.png",
        color_map = color_map,
    )

    # ── t-SNE 3D ──────────────────────────────────────────────────────────────
    plot_tsne_3d_html(
        D, labels,
        title         = f"t-SNE 3D — {run_label} mcs={min_cluster_size}",
        out_dir       = out_dir,
        filename_html = f"tsne3d_mcs{min_cluster_size}.html",
        color_map     = color_map,
    )

    # ── Outliers internes ──────────────────────────────────────────────────────
    describe_outliers_internal(
        df_sub    = df_sub,
        labels    = labels,
        run_label = f"{run_label}_mcs{min_cluster_size}",
        out_dir   = os.path.join(out_dir, f"outliers_internal_mcs{min_cluster_size}"),
    )

    log.info(f"[{run_label}] ✅ Rerun mcs={min_cluster_size} terminé")
    return df_sub, D, labels, clusterer

In [3]:
df = load_and_preprocess(CSV_PATH)

df_sub, D, labels, clusterer = rerun_postprocessing_no_sweep(
    df,
    scenario_name    = "scenario_2",
    run_label        = "s2_noweights",
    distance_metric  = "precomputed",
    min_cluster_size = 1900,
)

INFO | 
bio_exam_count distribution:
 bio_exam_count     n  pct
              0 12522 42.0
              1 12030 40.3
              2  4182 14.0
              3  1062  3.6
              4    43  0.1
INFO | 
imaging_exam_count distribution:
 imaging_exam_count     n  pct
                  0 14541 48.7
                  1 12968 43.5
                  2  2152  7.2
                  3   162  0.5
                  4    16  0.1
INFO | [s2_noweights] binary=12 | cat=0 | quanti=2 | n=29839
INFO | [s2_noweights] Distance matrix : 25.2s
INFO | [s2_noweights] HDBSCAN : 73.3s
INFO | [s2_noweights] clusters=7 | noise=6298 (21.1%)
INFO | Saved: Results/Regular_clustering/Full_dataset/With_counts/minmax_scaler/s2_noweights/heatmap_mcs1900.png
INFO | Saved: Results/Regular_clustering/Full_dataset/With_counts/minmax_scaler/s2_noweights/dendrogram_mcs1900.png
/home/nadia/pycharm_project_nad/.venv/lib/python3.12/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform w

split dataset ou pas split dataset ???

Ma recommandation — les deux approches répondent à des questions différentes :

"Quels profils de patients existent dans mes urgences ?" → pondération, tu gardes tout le monde ensemble

"Les patterns de prescription diffèrent-ils selon l'issue ?" → split, tu compares les deux populations

Pour une thèse/publication, faire les deux et comparer est souvent la meilleure approche.




Notes pondérations:

WEIGHT_DISPOSITION      = 5.0
WEIGHT_QUANTI           = 5.0
WEIGHT_IMAGING_BOOL     = 2.0   # ─┐ identiques
WEIGHT_BIO_EXAMS        = 2.0   # ─┘
WEIGHT_PROCEDURE        = 1.5   # légèrement en dessous
WEIGHT_BIO_VEINOUS      = 0.1
WEIGHT_IMAGING_DETAILED = 0.1


6  × IMAGING_COLS_BOOL      × 2.0  = 12.0  → 20%  ✅
4  × BIO_EXAMS              × 2.0  =  8.0  → 13%  ✅
1  × had_ekg                × 1.5  =  1.5  →  3%  ✅ (juste en dessous)
3  × DISPOSITION_COLS       × 5.0  = 15.0  → 25%  ✅
12 × IMAGING_COLS_DETAILED  × 0.1  =  1.2  →  2%  ✅
29 × BIO_VEINOUS            × 0.1  =  2.9  →  5%  ✅
2  × COLS_QUANTI            × 5.0  = 10.0  → 17%  ✅
─────────────────────────────────────────────────
Total                               = 50.6




ben c'est ca ma question a la bases c'est: il y a t'il des patterns de consommation de soins cachés dans mes données17:52Dans ce cas la réponse est claire — ni pondération forte, ni split.

Pourquoi ?
Ta question est exploratoire :
"Quels patterns CACHÉS existent ?"
→ tu ne sais pas à l'avance ce qui structure les données
→ tu veux laisser les données parler d'elles-mêmes
→ toute pondération forte = tu imposes une structure
→ splitter = tu présupposes que hosp est structurante
Si tu surpondères hospitalization à 5.0, tu vas forcément trouver des clusters séparés par l'hospitalisation — mais c'est toi qui l'as décidé, pas les données.

Ce que je ferais
Étape 1 — Clustering sans aucune pondération
          → voir ce qui émerge naturellement
          → est-ce que hospitalization structure déjà les clusters ?
          → est-ce que c'est la bio qui domine ?

Étape 2 — Sous-pondérer uniquement BIO_VEINOUS et IMAGING_DETAILED
          → pas pour imposer une structure
          → juste pour corriger le déséquilibre numérique
          → 29 variables bio = 51% sans correction, c'est un biais technique

Étape 3 — Comparer avec/sans sous-pondération
          → si les clusters changent beaucoup = la bio dominait artificiellement
          → si les clusters sont stables = les patterns sont robustes

La sous-pondération de BIO_VEINOUS et IMAGING_DETAILED est légitime méthodologiquement car c'est une correction technique, pas un choix analytique — tu corriges un déséquilibre de représentation, tu n'imposes pas une structure.
Tout le reste — pondérer disposition, ECG, quanti — c'est de l'analyse confirmatoire, pas exploratoire. À garder pour une deuxième phase une fois que tu as vu ce qui émerge naturellement.




Bonne façon de formuler l'objectif — chaque groupe pèse 1, pas chaque variable.
Donc :
1 × had_ekg              × 1.0  = 1.0
3 × DISPOSITION_COLS     × 1.0  = 3.0   (1 par variable)
6 × IMAGING_COLS_BOOL    × 1.0  = 6.0   (1 par variable)
4 × BIO_EXAMS            × 1.0  = 4.0   (1 par variable)
2 × COLS_QUANTI          × 1.0  = 2.0   (1 par variable)

# Les deux groupes à corriger :
29 × BIO_VEINOUS         × ?    = 1.0   → poids = 1/29 ≈ 0.034
12 × IMAGING_COLS_DETAILED × ?  = 1.0   → poids = 1/12 ≈ 0.083
Dans la config :
python# Correction technique uniquement — rééquilibrage numérique
WEIGHT_BIO_VEINOUS      = 1 / len(BIO_VEINOUS)         # ≈ 0.034
WEIGHT_IMAGING_DETAILED = 1 / len(IMAGING_COLS_DETAILED) # ≈ 0.083
Et dans run_hdbscan, bloc scenario_3 :
pythonelif scenario_name == "scenario_3":
    if use_group_weights:
        for col in col_list:
            idx_col = col_list.index(col)
            if col in BIO_VEINOUS:
                weights[idx_col] = WEIGHT_BIO_VEINOUS
            elif col in IMAGING_COLS_DETAILED:
                weights[idx_col] = WEIGHT_IMAGING_DETAILED
            # tout le reste reste à 1.0 — pas de pondération analytique
Ce qui donne :
1  × had_ekg               × 1.0    =  1.0  →  6%
3  × DISPOSITION_COLS      × 1.0    =  3.0  → 18%
6  × IMAGING_COLS_BOOL     × 1.0    =  6.0  → 35%
4  × BIO_EXAMS             × 1.0    =  4.0  → 24%
2  × COLS_QUANTI           × 1.0    =  2.0  → 12%
29 × BIO_VEINOUS           × 0.034  =  1.0  →  6%  ✅ comme had_ekg
12 × IMAGING_COLS_DETAILED × 0.083  =  1.0  →  6%  ✅ comme had_ekg
─────────────────────────────────────────────────────
Total                                = 18.0






POUR SCENARIO 2 SCORE DE SILOHOUETTE PLUS ELEVÉ AVEC PONDERATION QUE SANS PONDERATION:
✅ La pondération × 3 crée effectivement
   des clusters structurés autour de l'issue
   → preuve que tu "forçais" une structure

✅ Le clustering sans pondération est moins
   "propre" mathématiquement mais plus riche
   cliniquement — il découvre des patterns
   que tu ne connaissais pas à l'avance

✅ Argument supplémentaire pour justifier
   le rejet de la pondération dans ta thèse :
   "Un silhouette artificiellement élevé
    obtenu en surpondérant une variable d'issue
    ne reflète pas la richesse des patterns
    de consommation sous-jacents"


1. Sans pondération → hospitalization structure
                      naturellement les clusters
                      (dispersés dans 4-5 clusters)

2. Avec pondération × 3 → silhouette meilleur
                          → clusters séparés par hosp
                          → mais artificiel

3. Conclusion → hospitalization EST une variable
                structurante dans tes données
                → elle influence les patterns
                  de consommation


Si hospitalization structure tes clusters
→ tu ne peux pas l'ignorer
→ deux options :

Option A — pondérer          → artificiel, biais ❌
Option B — splitter          → propre, honnête ✅

"Nous avons observé que l'issue clinique
 (hospitalisation) influence les patterns
 de consommation de soins. Nous avons donc
 étudié séparément les deux populations
 pour identifier des patterns purs au sein
 de chaque groupe."



Étape 1 — Clustering global sans pondération
          → montre que hosp structure les données
          → silhouette < avec pondération
          → patterns moins nets mais réels

Étape 2 — Test de pondération × 3
          → silhouette artificiel
          → tous les hospitalisés dans 1 cluster
          → rejeté car confirmatoire

Étape 3 — Split hospit / non hospit
          → justifié par les observations 1 et 2
          → permet des patterns purs
          → meilleur silhouette attendu
          → cliniquement plus interprétable